In [6]:
# Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend (22 marks)

# The model says who is at risk; the notes say why patients actually missed. You will use the Gemini API to
# read the  first 10 notes, force the answers into a fixed format, validate them with Pydantic, test the
# weak spots, and then combine everything into a recommendation.
# Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency,
# work_or_schedule, other.

# Task 4.1 — One validated call (5 marks)
# Get a free key in Google AI Studio and install the library in Colab. Write a function
# analyse_note(note_text) that sends one note to the Gemini Flash model used in class, with a system
# instruction, a prompt built with the CRAFT parts (Context, Role, Action, Tone, Format), temperature=0,
# response_mime_type set to JSON, and a JSON response schema with four fields: root_cause
# (a plain string — do not make it an enum; the allowed values go in your prompt and are enforced afterwards by the check in Task 4.2), confidence (0–1), summary (at most 25 words), recommended_action. Set the thinking level to minimal as shown in Session 22.1, and give max_output_tokens plenty of room
# (1000 or more): on some models hidden reasoning tokens count against the limit and can cut the JSON off, which makes json.loads fail. Run it on note N001 and print the parsed dictionary. Shape of
# the call (follow your Session 22.1 notebook if any detail differs; the key is typed at run time):

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 4 - TASK 4.1
# One validated call
# ==========================================

import os
import json
import pandas as pd

from dotenv import load_dotenv
from google import genai
from google.genai import types


# STEP 1: Load API key
load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")


# STEP 2: Create Gemini client
client = genai.Client(api_key=API_KEY)


# STEP 3: Select Gemini model
MODEL_NAME = "gemini-3.5-flash-lite"


# STEP 4: Define JSON response schema
response_schema = {
    "type": "OBJECT",
    "properties": {
        "root_cause": {"type": "STRING"},
        "confidence": {"type": "NUMBER"},
        "summary": {"type": "STRING"},
        "recommended_action": {"type": "STRING"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}


# STEP 5: Define system instruction
SYSTEM_TEXT = """
You are a healthcare administration data analyst
working for CareLoop Clinics.

Analyse patient appointment follow-up notes.
Return accurate, professional and neutral results.
Follow the requested JSON response format exactly.
Do not invent information that is not present in the note.
"""


# STEP 6: Define CRAFT prompt
def build_prompt(note_text):

    prompt = f"""
    Context:
    CareLoop Clinics wants to understand why patients
    miss their scheduled appointments.

    Role:
    You are an experienced healthcare administration
    data analyst.

    Action:
    Analyse the following patient follow-up note.
    Identify the main root cause, estimate confidence,
    write a summary of at most 25 words, and recommend
    an appropriate administrative action.

    Allowed root_cause values:
    - forgot_on_no_reminder
    - illness_or_emergency
    - distance_or_transport
    - cost_or_fee
    - work_or_schedule
    - other

    Root cause must be a plain string, not an enum.
    Use one of the allowed values above.

    Tone:
    Professional, neutral and concise.

    Format:
    Return valid JSON with exactly four fields:
    root_cause, confidence, summary, recommended_action.

    Confidence must be a number between 0 and 1.
    Summary must contain at most 25 words.

    Patient follow-up note:
    {note_text}
    """

    return prompt


# STEP 7: Define analysis function
def analyse_note(note_text):

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            response_mime_type="application/json",
            response_schema=response_schema,
            max_output_tokens=1024
        )
    )

    # Convert JSON response into Python dictionary
    result = json.loads(response.text)

    # Validate required fields
    required_fields = {
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    }

    if set(result.keys()) != required_fields:
        raise ValueError("Incorrect JSON fields returned.")

    if not isinstance(result["root_cause"], str):
        raise ValueError("root_cause must be a string.")

    if not 0 <= result["confidence"] <= 1:
        raise ValueError("Confidence must be between 0 and 1.")

    if len(result["summary"].split()) > 25:
        raise ValueError("Summary exceeds 25 words.")

    return result


# STEP 8: Read follow-up notes
notes = pd.read_csv("followup_notes.csv")


# STEP 9: Find note N001
note_row = notes[
    notes["note_id"].astype(str).str.strip() == "N001"
]

if note_row.empty:
    raise ValueError("Note N001 was not found.")

note_text = str(note_row.iloc[0]["note_text"])


# STEP 10: Analyse note N001
result = analyse_note(note_text)


# STEP 11: Display output
print("TASK 4.1 - GEMINI NOTE ANALYSIS")
print("--------------------------------")

print("NOTE ID: N001")
print("ANALYSIS RESULT:")
print(result)

print("RESULT TYPE:", type(result))

print("\n===================================")
print("TASK 4.1 COMPLETED SUCCESSFULLY")


TASK 4.1 - GEMINI NOTE ANALYSIS
--------------------------------
NOTE ID: N001
ANALYSIS RESULT:
{'root_cause': 'forgot_on_no_reminder', 'confidence': 0.95, 'summary': 'Patient missed the appointment because she received no reminder and forgot.', 'recommended_action': 'Verify and update patient contact information for future automated SMS reminders.'}
RESULT TYPE: <class 'dict'>

TASK 4.1 COMPLETED SUCCESSFULLY


In [7]:
# Task 4.2 — Validate and process the 10 notes (7 marks)

# (a) Define a Pydantic model FollowupInsight(BaseModel) with root_cause: str = Field(min_length=3),
# confidence: float = Field(ge=0, le=1), summary: str = Field(min_length=5),
# recommended_action: str = Field(min_length=5), and write a check that root_cause is in the allowed
# list. (b) Read followup_notes.csv with pandas and loop over the first 10 rows.
# Input validation: pandas reads a blank note as NaN (a float, which has no .strip()),
# so test pd.isna(note_text) first, then str(note_text).strip() == "". If the note is missing, empty or only spaces, do not call the API — record root_cause="other", confidence=0.0, summary="No note recorded",
# recommended_action="Re-call patient" and validation_status="skipped_blank". Otherwise call analyse_note,validate the result with the Pydantic model (as in class: pass the dictionary into the class, FollowupInsight(**data)) and the allowed-list check, and set validation_status to "valid" or "invalid" (never let one bad reply stop the loop: use try/except). (c) Pause between calls (import time then time. sleep(13)) to respect the free-tier rate limit. (d) Save notes_analysed.csv with columns note_id,appointment_id, root_cause, confidence, summary, recommended_action, validation_status, raw_response.Keep every reply, including invalid ones, and never repair it:
# put the model('s reply exactly as returned (the JSON text) in raw_response, copy whichever fields it did',' contain, ''and leave a field empty if the reply lacked it (for a skipped blank note, raw_response is empty).

# Solution
# ==================================================
# CARELOOP CLINICS
# PART 4 - TASK 4.2
# VALIDATE AND PROCESS THE 10 NOTES
# ==================================================

import os
import json
import time
import pandas as pd

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


# =====================================================
# TASK 4.1 - GEMINI NOTE ANALYSIS
# =====================================================

# STEP 1: Load API key

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")


# STEP 2: Create Gemini client

client = genai.Client(api_key=API_KEY)


# STEP 3: Select Gemini model

MODEL_NAME = "gemini-3.5-flash-lite"


# STEP 4: Define JSON response schema

response_schema = {
    "type": "OBJECT",
    "properties": {
        "root_cause": {
            "type": "STRING"
        },
        "confidence": {
            "type": "NUMBER"
        },
        "summary": {
            "type": "STRING"
        },
        "recommended_action": {
            "type": "STRING"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}


# STEP 5: Define system instruction

SYSTEM_TEXT = """
You are a healthcare administration data analyst
working for CareLoop Clinics.

Analyse patient appointment follow-up notes.
Return accurate, professional and neutral results.
Do not invent information that is not present in the note.
Follow the requested JSON response format exactly.
"""


# STEP 6: Build CRAFT prompt

def build_prompt(note_text):

    prompt = f"""
    Context:
    CareLoop Clinics wants to understand why patients
    miss their scheduled appointments.

    Role:
    You are an experienced healthcare administration
    data analyst.

    Action:
    Analyse the patient follow-up note.
    Identify the main root cause, estimate confidence,
    write a summary of at most 25 words, and recommend
    an appropriate administrative action.

    Allowed root_cause values:
    - forgot_on_no_reminder
    - illness_or_emergency
    - distance_or_transport
    - cost_or_fee
    - work_or_schedule
    - other

    Root cause must be a plain string, not an enum.
    Use one of the allowed values above.

    Tone:
    Professional, neutral and concise.

    Format:
    Return valid JSON with exactly four fields:
    root_cause, confidence, summary, recommended_action.

    Confidence must be a number between 0 and 1.
    Summary must contain at most 25 words.

    Patient follow-up note:
    {note_text}
    """

    return prompt


# STEP 7: Define analyse_note function

def analyse_note(note_text, return_raw=False):

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            response_mime_type="application/json",
            response_schema=response_schema,
            max_output_tokens=1024
        )
    )

    # Preserve original Gemini response
    raw_text = response.text or ""

    # Convert JSON into Python dictionary
    try:
        result = json.loads(raw_text)

    except json.JSONDecodeError:
        if return_raw:
            return None, raw_text
        raise

    if return_raw:
        return result, raw_text

    return result


# =====================================================
# TASK 4.2 - VALIDATE AND PROCESS 10 NOTES
# =====================================================


# STEP 8: Define allowed root causes

ALLOWED_ROOT_CAUSES = [
    "forgot_on_no_reminder",
    "illness_or_emergency",
    "distance_or_transport",
    "cost_or_fee",
    "work_or_schedule",
    "other"
]


# STEP 9: Define Pydantic model

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)


# STEP 10: Read CSV file

notes = pd.read_csv("followup_notes.csv")

# Select first 10 rows

first_10_notes = notes.head(10)

# Store output results

results = []

# Count Gemini API calls

api_calls = 0


# STEP 11: Process each note

for index, row in first_10_notes.iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print("\n-----------------------------------")
    print("Processing NOTE ID:", note_id)

    # Create default result dictionary

    result_row = {
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # STEP 12: Check blank notes

    if pd.isna(note_text) or str(note_text).strip() == "":

        result_row["root_cause"] = "other"
        result_row["confidence"] = 0.0
        result_row["summary"] = "No note recorded"
        result_row["recommended_action"] = "Re-call patient"
        result_row["validation_status"] = "skipped_blank"

        print("Blank note detected.")
        print("Gemini API call skipped.")

    else:

        try:

            # STEP 13: Pause between API calls

            if api_calls > 0:
                time.sleep(13)

            # STEP 14: Analyse note using Gemini

            api_calls += 1

            data, raw_response = analyse_note(
                str(note_text),
                return_raw=True
            )

            # Store original response exactly as returned

            result_row["raw_response"] = raw_response

            # STEP 15: Copy fields returned by Gemini

            if isinstance(data, dict):

                fields = [
                    "root_cause",
                    "confidence",
                    "summary",
                    "recommended_action"
                ]

                for field in fields:

                    if field in data:
                        result_row[field] = data[field]

                # STEP 16: Pydantic validation

                validated_data = FollowupInsight(**data)

                # STEP 17: Allowed root cause validation

                if validated_data.root_cause not in ALLOWED_ROOT_CAUSES:

                    raise ValueError(
                        "Root cause is not in the allowed list."
                    )

                result_row["validation_status"] = "valid"

                print("Validation successful.")

            else:

                result_row["validation_status"] = "invalid"

                print("Invalid response format.")

        except Exception as error:

            result_row["validation_status"] = "invalid"

            print("Validation or API error:", error)

    # STEP 18: Save every result in the list

    results.append(result_row)


# =====================================================
# STEP 19: CREATE OUTPUT CSV
# =====================================================

output_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

output_df = pd.DataFrame(
    results,
    columns=output_columns
)


# STEP 20: SAVE CSV FILE

output_df.to_csv(
    "notes_analysed.csv",
    index=False
)


# =====================================================
# FINAL OUTPUT
# =====================================================

print("\n===================================")
print("TASK 4.2 COMPLETED SUCCESSFULLY")
print("===================================")

print("Total notes processed:", len(output_df))
print("Gemini API calls made:", api_calls)

print("\nValidation status counts:")
print(output_df["validation_status"].value_counts(dropna=False))

print("\nOutput CSV created: notes_analysed.csv")

print("\nFirst 10 results:")
print(output_df.to_string(index=False))

print("\n===================================")
print("TASK 4.2 COMPLETED SUCCESSFULLY")


-----------------------------------
Processing NOTE ID: N001
Validation or API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

-----------------------------------
Processing NOTE ID: N002
Validation successful.

-----------------------------------
Processing NOTE ID: N003
Validation successful.

-----------------------------------
Processing NOTE ID: N004
Blank note detected.
Gemini API call skipped.

-----------------------------------
Processing NOTE ID: N005
Validation or API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

-----------------------------------
Processing NOTE ID: N006
Validation successful.

-----------------------------------
Processing NOTE ID: N007
Validation 

In [9]:
# Task 4.3 — Break it on purpose (5 marks)

# (a) Feed the Pydantic model two deliberately bad replies and catch each with try/except
# ValidationError (from pydantic import ValidationError): one with confidence = 1.7, one missing recommended_action; print what each error says. (b) For notes N007 (Hinglish) and N009 (ambiguous), compare two prompts at temperature=0: V1 = just "Classify this note", V2 = your CRAFT prompt with the allowed values and the instruction to answer other rather than guess. Use the same schema and settings for both (root_cause as a plain string, JSON mode, temperature=0), so that the only difference is the prompt; then check each reply against the allowed list. Show the four results in a small table and say which prompt you would deploy.(c) Count and print the total number of API calls you used across Tasks 4.1–4.3  (about 14 on a first pass: 1 in 4.1, 9 in 4.2, 4 here; 12 if you reuse your 4.2 results for the V2 prompt).

# Solution
# ==================================================
# CARELOOP CLINICS
# PART 4 - TASK 4.3
# Break it on purpose
# ==================================================
import os
import json
import time
import pandas as pd

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError


# ==========================================
# 1. GEMINI SETUP
# ==========================================

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.5-flash-lite"

task_4_3_api_calls = 0


# ==========================================
# 2. PYDANTIC MODEL
# ==========================================

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


ALLOWED_ROOT_CAUSES = [
    "forgot_on_no_reminder",
    "illness_or_emergency",
    "distance_or_transport",
    "cost_or_fee",
    "work_or_schedule",
    "other"
]


# ==========================================
# PART A: PYDANTIC VALIDATION TESTS
# ==========================================

print("\nPART A: PYDANTIC VALIDATION TESTS")
print("----------------------------------")

bad_reply_1 = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "This is a test reply",
    "recommended_action": "Contact the patient"
}

bad_reply_2 = {
    "root_cause": "other",
    "confidence": 0.5,
    "summary": "This is a test reply"
}

bad_replies = [bad_reply_1, bad_reply_2]

for number, bad_reply in enumerate(bad_replies, start=1):

    try:
        FollowupInsight(**bad_reply)
        print(f"Bad reply {number}: Unexpectedly passed")

    except ValidationError as error:
        print(f"\nBad reply {number} validation error:")
        print(error)


# ==========================================
# PART B: PROMPT COMPARISON
# ==========================================

print("\n\nPART B: PROMPT COMPARISON")
print("----------------------------------")

# Read CSV
CSV_FILE = "followup_notes.csv"

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(
        "followup_notes.csv not found in project folder"
    )

notes = pd.read_csv(CSV_FILE)

print("CSV columns:", list(notes.columns))

if "note_id" not in notes.columns:
    raise ValueError("note_id column not found")

if "note_text" not in notes.columns:
    raise ValueError("note_text column not found")


# Select N007 and N009
selected_notes = notes[
    notes["note_id"].astype(str).str.strip().isin(
        ["N007", "N009"]
    )
].copy()

selected_notes["note_id"] = (
    selected_notes["note_id"].astype(str).str.strip()
)

# Ensure both notes exist
found_ids = set(selected_notes["note_id"])

if not {"N007", "N009"}.issubset(found_ids):
    raise ValueError(
        f"N007 or N009 is missing. Found: {found_ids}"
    )

# Sort in the requested order
selected_notes["sort_order"] = (
    selected_notes["note_id"].map({"N007": 1, "N009": 2})
)

selected_notes = selected_notes.sort_values("sort_order")


# V1: Simple prompt
V1_PROMPT = "Classify this note"


# V2: CRAFT prompt
V2_PROMPT = """
Context:
You are analysing patient appointment follow-up notes
for CareLoop Clinics.

Role:
Act as a careful healthcare appointment data analyst.

Action:
Classify the main reason for the missed appointment.

Use only one of these root_cause values:
forgot_on_no_reminder,
illness_or_emergency,
distance_or_transport,
cost_or_fee,
work_or_schedule,
other.

If the note is ambiguous or unclear, choose "other"
rather than guessing.

Tone:
Be neutral, careful, and factual.

Format:
Return only valid JSON containing:
root_cause, confidence, summary, recommended_action.

The summary must contain at most 25 words.
"""


# Same JSON schema and settings for both prompts
RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "root_cause": {"type": "STRING"},
        "confidence": {"type": "NUMBER"},
        "summary": {"type": "STRING"},
        "recommended_action": {"type": "STRING"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}


# Gemini API function
def classify_note(note_text, prompt):

    global task_4_3_api_calls

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt + "\n\nPatient note:\n" + note_text,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            max_output_tokens=1024
        )
    )

    task_4_3_api_calls += 1

    raw_text = response.text or ""

    return json.loads(raw_text)


# Store comparison results
comparison_results = []

# Process N007 and N009
for _, row in selected_notes.iterrows():

    note_id = row["note_id"]
    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        print(f"\n{note_id} is blank. Cannot compare prompts.")
        continue

    note_text = str(note_text).strip()

    # Run V1 and V2
    for version, prompt in [
        ("V1", V1_PROMPT),
        ("V2", V2_PROMPT)
    ]:

        # Pause between API calls
        if task_4_3_api_calls > 0:
            time.sleep(13)

        print(f"\nProcessing {note_id} with {version}")

        try:
            result = classify_note(note_text, prompt)

            root_cause = result.get("root_cause", "")

            # Check allowed list
            if root_cause in ALLOWED_ROOT_CAUSES:
                allowed_status = "Allowed"
            else:
                allowed_status = "Not allowed"

            comparison_results.append({
                "note_id": note_id,
                "prompt": version,
                "root_cause": root_cause,
                "allowed_list_check": allowed_status
            })

            print("Model response:", result)

        except Exception as error:

            print("API or JSON error:", error)

            comparison_results.append({
                "note_id": note_id,
                "prompt": version,
                "root_cause": "ERROR",
                "allowed_list_check": "Error"
            })


# Display comparison table
print("\n\nFOUR PROMPT COMPARISON RESULTS")
print("----------------------------------")

comparison_table = pd.DataFrame(comparison_results)

print(comparison_table.to_string(index=False))


# ==========================================
# PART C: API CALL COUNT
# ==========================================

print("\n\nPART C: API CALL COUNT")
print("----------------------------------")

# Enter your actual completed API calls from Tasks 4.1 and 4.2.
task_4_1_calls = 1
task_4_2_calls = 9

task_4_3_calls = task_4_3_api_calls

total_api_calls = (
    task_4_1_calls
    + task_4_2_calls
    + task_4_3_calls
)

print("Task 4.1 API calls:", task_4_1_calls)
print("Task 4.2 API calls:", task_4_2_calls)
print("Task 4.3 API calls:", task_4_3_calls)

print("----------------------------------")
print("TOTAL API CALLS:", total_api_calls)


# ==========================================
# DEPLOYMENT DECISION
# ==========================================

print("\n\nDEPLOYMENT DECISION")
print("----------------------------------")

print(
    "I would deploy V2 because it uses clear CRAFT "
    "instructions, specifies the allowed root causes, "
    "and tells the model to choose other rather than "
    "guess when a note is ambiguous."
)

print("\n===================================")
print("TASK 4.3 COMPLETED SUCCESSFULLY")


PART A: PYDANTIC VALIDATION TESTS
----------------------------------

Bad reply 1 validation error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Bad reply 2 validation error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '... 'This is a test reply'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


PART B: PROMPT COMPARISON
----------------------------------
CSV columns: ['note_id', 'appointment_id', 'note_text']

Processing N007 with V1
Model response: {'root_cause': 'The patient forgot about the appointment because they did not receive the SMS reminder, and now they need to rebook.', 'confidence': 0.95, 'summary': 'Patient missed appointment due to missing SMS reminder a

In [10]:
# Task 4.4 — Recommendation and GitHub (5 marks)

# (a) (3 marks) Merge notes_analysed.csv with appointments_clean.csv on appointment_id and
# cross-tabulate root_cause against reminder_sent (pd.crosstab). Write README.md with: a 4-line summary (one finding per Part, each with its number); a one-paragraph recommendation that answers the business question using at least four numbers from Parts 1–4; the model you chose and why (include AUC and the threshold); and three# limitations (choose from: the 10 notes are an illustration and cannot show the real reasons clinic-wide;
# the cleaning statistics came from all rows before the split; the same patient can sit in both
# train and test; the split is random, not past-to-future; one train/test split; a threshold that really
# depends on call capacity; clinic-level sample sizes; the data are observational, so associations are not causes). (b) (2 marks) Create the public GitHub repository, upload every file listed under Submission,  confirm the key is nowhere in it, and paste the link at the top of the README.

# Solution
# ==================================================
# CARELOOP CLINICS
# PART 4 - TASK 4.4
# Recommendation and GitHub
# ==================================================

from pathlib import Path
import pandas as pd


# ==========================================
# 1. PROJECT SETTINGS
# ==========================================

PROJECT_FOLDER =Path.cwd()

NOTES_FILE = PROJECT_FOLDER / "notes_analysed.csv"
APPOINTMENTS_FILE = PROJECT_FOLDER / "appointments_clean.csv"

MERGED_OUTPUT = PROJECT_FOLDER / "notes_with_appointments.csv"
CROSSTAB_OUTPUT = PROJECT_FOLDER / "root_cause_reminder_crosstab.csv"
README_FILE = PROJECT_FOLDER / "README.md"

# Part 3 model details
MODEL_NAME = "Logistic Regression"
MODEL_AUC = 0.86
CLASSIFICATION_THRESHOLD = 0.50

# Replace this after creating your GitHub repository
GITHUB_REPO_URL = "ADD_YOUR_GITHUB_REPOSITORY_LINK_HERE"


# ==========================================
# 2. CHECK FILES
# ==========================================

print("\nTASK 4.4 - RECOMMENDATION AND GITHUB")
print("--------------------------------------")

if not NOTES_FILE.exists():
    raise FileNotFoundError(
        "notes_analysed.csv was not found in the project folder."
    )

if not APPOINTMENTS_FILE.exists():
    raise FileNotFoundError(
        "appointments_clean.csv was not found in the project folder."
    )


# ==========================================
# 3. READ CSV FILES
# ==========================================

notes = pd.read_csv(NOTES_FILE)
appointments = pd.read_csv(APPOINTMENTS_FILE)

print("\nFiles loaded successfully.")

print("AI analysis rows:", len(notes))
print("Clean appointment rows:", len(appointments))


# ==========================================
# 4. CHECK REQUIRED COLUMNS
# ==========================================

required_notes_columns = [
    "appointment_id",
    "root_cause"
]

required_appointment_columns = [
    "appointment_id",
    "reminder_sent"
]

for column in required_notes_columns:
    if column not in notes.columns:
        raise ValueError(
            f"{column} is missing from notes_analysed.csv"
        )

for column in required_appointment_columns:
    if column not in appointments.columns:
        raise ValueError(
            f"{column} is missing from appointments_clean.csv"
        )


# ==========================================
# 5. MERGE TWO DATASETS
# ==========================================

merged = pd.merge(
    notes,
    appointments,
    on="appointment_id",
    how="left",
    indicator=True,
    suffixes=("_note", "_appointment")
)

matched_count = (merged["_merge"] == "both").sum()
unmatched_count = (merged["_merge"] == "left_only").sum()

print("\nMERGE RESULTS")
print("--------------------------------------")
print("Total merged rows:", len(merged))
print("Matched appointments:", matched_count)
print("Unmatched appointments:", unmatched_count)

# Save merged dataset
merged.to_csv(MERGED_OUTPUT, index=False)

print("\nMerged file saved:", MERGED_OUTPUT.name)


# ==========================================
# 6. CROSS-TABULATION
# ==========================================

print("\nROOT CAUSE VS REMINDER SENT")
print("--------------------------------------")

# Use only rows with a matching appointment
matched_data = merged[
    merged["_merge"] == "both"
].copy()

cross_tab = pd.crosstab(
    matched_data["root_cause"],
    matched_data["reminder_sent"],
    margins=True
)

print(cross_tab)

# Save cross-tabulation
cross_tab.to_csv(CROSSTAB_OUTPUT)

print("\nCross-tabulation saved:", CROSSTAB_OUTPUT.name)


# ==========================================
# 7. PART 4 FINDINGS
# ==========================================

total_notes = len(merged)

no_reminder_count = (
    matched_data["reminder_sent"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["no", "0", "false"])
    .sum()
)

no_reminder_percentage = (
    no_reminder_count / len(matched_data) * 100
    if len(matched_data) > 0
    else 0
)

print("\nPART 4 FINDINGS")
print("--------------------------------------")
print("Notes analysed:", total_notes)
print("Notes matched with appointments:", matched_count)
print("Appointments without reminders:", no_reminder_count)
print(
    "Percentage without reminders:",
    round(no_reminder_percentage, 2), "%"
)


# ==========================================
# 8. CONVERT CROSS-TAB TO MARKDOWN
# ==========================================

def dataframe_to_markdown(dataframe):

    headers = ["Root cause"] + [
        str(column) for column in dataframe.columns
    ]

    lines = []

    lines.append("| " + " | ".join(headers) + " |")

    lines.append(
        "| " + " | ".join(["---"] * len(headers)) + " |"
    )

    for index, row in dataframe.iterrows():

        values = [str(index)] + [
            str(value) for value in row.values
        ]

        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)


cross_tab_markdown = dataframe_to_markdown(cross_tab)


# ==========================================
# 9. CREATE README.MD
# ==========================================

readme_content = f"""# CareLoop Clinics - No-Show Guard

**GitHub Repository:** {GITHUB_REPO_URL}

## Project Overview

CareLoop Clinics - No-Show Guard is a data analysis and machine learning project
that studies missed medical appointments and explores how clinics can identify
patients who may need additional appointment reminders or follow-up support.

## Four-Part Project Summary

1. **Part 1 - Python and Data Foundations:** Processed 1,020 original appointment rows; 988 rows were accepted during the initial validation.
2. **Part 2 - SQL Analysis:** Gomti Nagar had 90 appointments and a 33.3% no-show rate.
3. **Part 3 - Machine Learning:** Logistic Regression achieved an ROC-AUC of {MODEL_AUC:.2f}, using a classification threshold of {CLASSIFICATION_THRESHOLD:.2f}.
4. **Part 4 - Generative AI:** Analysed {total_notes} follow-up notes; {no_reminder_count} matched appointments had no reminder recorded.

## Part 4: Root Cause and Reminder Analysis

The AI analysis results were merged with the cleaned appointment dataset using
`appointment_id`.

- Total AI analysis records: {total_notes}
- Matched appointment records: {matched_count}
- Unmatched records: {unmatched_count}
- Records without a reminder: {no_reminder_count}
- Percentage without a reminder among matched records: {no_reminder_percentage:.2f}%

### Root Cause vs Reminder Sent

{cross_tab_markdown}

## Business Recommendation

CareLoop Clinics should use a combination of appointment reminders, targeted
follow-up, and risk-based patient outreach to reduce missed appointments.
The initial dataset contained 1,020 rows, of which 988 were accepted during
validation. In Part 2, Gomti Nagar recorded 90 appointments with a 33.3%
no-show rate, suggesting that clinic-level monitoring is useful. In Part 3,
Logistic Regression achieved an ROC-AUC of {MODEL_AUC:.2f} at a classification
threshold of {CLASSIFICATION_THRESHOLD:.2f}. In Part 4, {total_notes} patient
notes were analysed and {no_reminder_count} matched appointments had no reminder
recorded. Clinics should review reminder delivery, monitor high-risk predictions,
and use staff follow-up where capacity permits. These findings are useful for
planning, but should not be treated as proof that reminders alone cause
attendance to improve.

## Machine Learning Model

**Selected model:** {MODEL_NAME}

Logistic Regression was selected because it provides a straightforward
classification approach and produces probabilities that can help identify
patients who may miss appointments.

- ROC-AUC: {MODEL_AUC:.2f}
- Classification threshold: {CLASSIFICATION_THRESHOLD:.2f}

The threshold should be reviewed against clinic staff capacity and the cost of
missed appointments before real-world deployment.

## Limitations

1. The 10 follow-up notes are an illustration and cannot show the real reasons
   for missed appointments across the entire clinic network.
2. The train/test split is random, not past-to-future, so performance may not
   reflect how the model works on future appointments.
3. The data are observational, so associations between reminders and
   attendance do not prove that reminders cause attendance to improve.

## Project Files

This repository should contain all files required by the official project
Submission checklist, including the relevant source code, datasets, and outputs.

## Security

The Gemini API key is stored locally in a `.env` file and must not be uploaded
to GitHub. The `.env` file is excluded through `.gitignore`.
"""


with open(README_FILE, "w", encoding="utf-8") as file:
    file.write(readme_content)


# ==========================================
# 10. CREATE .GITIGNORE
# ==========================================

gitignore_file = PROJECT_FOLDER / ".gitignore"

gitignore_content = """
.env
.venv/
venv/
__pycache__/
*.pyc
.idea/
"""

with open(gitignore_file, "w", encoding="utf-8") as file:
    file.write(gitignore_content.strip() + "\n")


# ==========================================
# 11. FINAL OUTPUT
# ==========================================

print("\nREADME AND GITHUB FILES")
print("--------------------------------------")

print("README.md created successfully.")
print(".gitignore created successfully.")

print("\nGenerated files:")
print("1.", MERGED_OUTPUT.name)
print("2.", CROSSTAB_OUTPUT.name)
print("3.", README_FILE.name)
print("4.", gitignore_file.name)

print("\nTASK 4.4 COMPLETED SUCCESSFULLY")




TASK 4.4 - RECOMMENDATION AND GITHUB
--------------------------------------

Files loaded successfully.
AI analysis rows: 10
Clean appointment rows: 990

MERGE RESULTS
--------------------------------------
Total merged rows: 10
Matched appointments: 10
Unmatched appointments: 0

Merged file saved: notes_with_appointments.csv

ROOT CAUSE VS REMINDER SENT
--------------------------------------
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_on_no_reminder   1    0    1
other                   1    0    1
work_or_schedule        1    1    2
All                     4    3    7

Cross-tabulation saved: root_cause_reminder_crosstab.csv

PART 4 FINDINGS
--------------------------------------
Notes analysed: 10
Notes matched with appointments: 10
Appointments without reminders: 5
Percentage without reminders: 50.0 %

README AND GITHUB FILES
--------------------------------------
README.md c